# Une recherche plus profonde

## Le retrieval, c'est quoi ?
Le retrieval concerne la recherche d'information par similarité. Avec une photo requête et une base de photos de référence/le catalogue, on cherche à renvoyer les top-1 et top-10 des individus candidats les plus probables. On classe des candidats pour les proposer (top-1/top-10).

Ce notebook propose et teste plusieurs solutions pour améliorer le retrieval/le flux de recherche dans le catalogue :
1. Chamfer distance vs. max() pour comparer les cosinus
2. Matching local (LightGlue/ALIKED) / Re-ranking
3. Calibration par régression logistique
4. Utilisation du contour matching (CurvRank/FinFindR)



## Step 0 - Initialisation & chargement des données

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from dorsalmatch.chamfer import MODES, cosine_similarity_matrix, rank  # noqa: E402

DATA = ROOT / "data"
HISTORIQUES = ["2026-10-29_miraceti", "2026-11-06_bestof_20230307_sete"]
CATALOGUE = "2026-11-06_catalogue_tt"
MODELS = ["cetaceanet"]
TOP = 10
NEAR_IDENTICAL = 0.97  # au-delà, c'est la même photo.
CASES = ["bien rangé", f"top {TOP}, mal rangé", f"hors top {TOP}"]

In [ ]:
def load(kind, delivery, model):
    folder = DATA / "derived" / "embeddings" / kind / delivery
    photos = pd.read_csv(folder / "photos.csv", dtype=str, keep_default_na=False)
    return photos.to_dict("records"), np.load(folder / f"{model}.npy")


def load_all(model):
    history, vectors = [], []
    for delivery in HISTORIQUES:
        photos, embeddings = load("historique", delivery, model)
        history.extend([{**photo, "livraison": delivery} for photo in photos])
        vectors.append(embeddings)
    catalogue, catalog_embeddings = load("catalogue", CATALOGUE, model)
    return history, np.vstack(vectors), catalogue, catalog_embeddings


DATA_BY_MODEL = {model: load_all(model) for model in MODELS}

In [ ]:
reference = MODELS[0]
history, history_embeddings, catalogue, catalogue_embeddings = DATA_BY_MODEL[reference]


def findable(line):
    photo = history[line]
    same = [
        i
        for i, other in enumerate(catalogue)
        if other["individu"] == photo["individu"] and other["cote"] == photo["cote"]
    ]
    if not same:
        return "côté absent du catalogue"
    similarities = cosine_similarity_matrix(
        history_embeddings[line : line + 1],
        catalogue_embeddings[same],
        near_identical=None,
    )[0]
    if np.all(similarities >= NEAR_IDENTICAL):
        return "c'est la photo du catalogue"
    return None


REASONS = [findable(i) for i in range(len(history))]
print(f"Historique : {len(history)} photos ; catalogue : {len(catalogue)} photos")
print(f"Photos évaluables : {sum(reason is None for reason in REASONS)}")

## Step 1 - Chamfer distance

Chamfer agrége les similarités entre les ensembles d’embeddings des photos requête et catalogue : il compare les nuages de vecteurs/embeddings des photos requête et catalogue pour calculer les similarités d’embeddings.

Donc, Chamfer Cosine Similarity (symétrique ou asymétrique) évalue la proximité globale entre deux nuages de vecteurs, i.e. la moyenne des meilleurs cosinus mutuels requête/catalogue.

Pour chaque côté, on calcule les cosinus entre toutes les photos de la requête et du catalogue. Les paires dont le cosinus atteint `NEAR_IDENTICAL` sont masquées avant les réductions.

- `max` : maximum de toutes les paires (référence).
- `chamfer_asym` : moyenne, sur les photos requêtes, de leur meilleur match catalogue
- `chamfer_sym` : moyenne du Chamfer asymétrique et de son inverse

Lors d’une reconstruction par dauphin, toutes ses requêtes valides sont regroupées. Les scores sont d’abord calculés séparément pour D et G, puis le maximum entre les côtés est le score final du candidat, comme dans le notebook source.

In [ ]:
def reconstruct(individual, lines, mode, photos, vectors, catalog, catalog_vectors):
    ranked, _ = rank(
        [photos[i] for i in lines], vectors[lines], catalog, catalog_vectors, mode
    )
    labels = [c["individu"] for c in ranked]
    position = labels.index(individual) if individual in labels else None
    return {
        "individu": individual,
        "lines": lines,
        "candidats": ranked,
        "rang": position + 1 if position is not None else None,
    }


def case(result):
    if result["rang"] == 1:
        return CASES[0]
    if result["rang"] and result["rang"] <= TOP:
        return CASES[1]
    return CASES[2]


eligible = [i for i, reason in enumerate(REASONS) if reason is None]
BY_PHOTO, BY_DOLPHIN = {}, {}

for model, (photos, vectors, catalog, catalog_vectors) in DATA_BY_MODEL.items():
    lines_by_dolphin = {}
    for i in eligible:
        lines_by_dolphin.setdefault(photos[i]["individu"], []).append(i)
    for mode in MODES:
        args = (mode, photos, vectors, catalog, catalog_vectors)
        BY_PHOTO[mode, model] = [
            reconstruct(photos[i]["individu"], [i], *args) for i in eligible
        ]
        BY_DOLPHIN[mode, model] = [
            reconstruct(name, lines, *args)
            for name, lines in sorted(lines_by_dolphin.items())
        ]


def counts(results, title):
    print(title)
    for name in CASES:
        number = sum(case(r) == name for r in results)
        print(f"  {name:<22} {number:>4}  ({number / len(results):.0%})")
    missed = [r["rang"] for r in results if case(r) == CASES[2] and r["rang"]]
    if missed:
        print(f"  rang médian hors top {TOP} : {np.median(missed):.0f}")
        for limit in (20, 50, 100):
            print(f"  dans le top {limit:<4} {sum(rank_ <= limit for rank_ in missed)}")
    absent = sum(r["rang"] is None for r in results)
    if absent:
        print(f"  dauphin absent des candidats : {absent}")


for mode in MODES:
    for model in MODELS:
        counts(BY_PHOTO[mode, model], f"{mode} - {model} - par photo")
        counts(BY_DOLPHIN[mode, model], f"{mode} - {model} - par dauphin")

In [ ]:
for model in MODELS:
    for level, table in (("photo", BY_PHOTO), ("dauphin", BY_DOLPHIN)):
        base = [r["rang"] for r in table["max", model]]
        for mode in MODES[1:]:
            pairs = [
                (c["rang"], b)
                for c, b in zip(table[mode, model], base, strict=True)
                if c["rang"] is not None and b is not None
            ]
            new, old = np.array(pairs).T
            print(
                f"{model} - {level} - {mode} vs max (n={len(pairs)}) : "
                f"mieux {np.sum(new < old)} / égal {np.sum(new == old)} "
                f"/ moins bien {np.sum(new > old)}"
            )

Une photo de l'historique, face aux 10 dauphins proposés par le ML : **vert**, le bon dauphin ; **rouge**, un autre. Le chiffre est le score.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from dorsalmatch.chamfer import SIDES


def photo_path(kind, delivery, photo):
    return (
        DATA
        / "curated"
        / kind
        / "photos"
        / delivery
        / photo["individu"]
        / photo["fichier_origine"]
    )


SIDES_NAMES = {"D": "droite", "G": "gauche"}
# Case grise : pas de photo pour ce côté, comme dans la démo
MISSING = np.full((260, 400, 3), 225, dtype=np.uint8)


def catalogue_photo(individual, side):
    return next(
        (
            photo
            for photo in catalogue
            if photo["individu"] == individual and photo["cote"] == side
        ),
        None,
    )


def small(path):
    image = Image.open(path).convert("RGB")
    image.thumbnail((400, 400))
    return image


def show(result, history, catalogue, prefix=""):
    query = history[result["lines"][0]]
    candidates = result["candidats"][:TOP]
    fig = plt.figure(figsize=(15, 9), dpi=80)
    grid = fig.add_gridspec(
        4,
        6,
        width_ratios=[2.4, 1, 1, 1, 1, 1],
        wspace=0.08,
        hspace=0.45,
        top=0.9,
        bottom=0.02,
    )
    ax = fig.add_subplot(grid[:, 0])
    ax.imshow(small(photo_path("historique", query["livraison"], query)))
    ax.set_title(
        f"{prefix}{query['individu']} · {query['cote']} · rang {result['rang']}",
        loc="left",
        fontsize=12,
    )
    ax.axis("off")
    for position, candidate in enumerate(candidates):
        good = candidate["individu"] == query["individu"]
        for side_line, side in enumerate(SIDES):
            ax = fig.add_subplot(
                grid[(position // 5) * 2 + side_line, 1 + position % 5]
            )
            photo = next(
                (
                    p
                    for p in catalogue
                    if p["individu"] == candidate["individu"] and p["cote"] == side
                ),
                None,
            )
            ax.imshow(
                small(photo_path("catalogue", CATALOGUE, photo)) if photo else MISSING
            )
            score = f"{candidate[side]:.2f}" if side in candidate else "–"
            label = (
                f"{SIDES_NAMES[side]} {score}"
                if photo
                else f"pas de photo {SIDES_NAMES[side]}"
            )
            title = (
                f"{position + 1}. {candidate['individu']} · "
                f"{candidate['score']:.2f}\n{label}"
                if side_line == 0
                else label
            )
            ax.set_title(title, fontsize=9, color="green" if good else "red")
            ax.axis("off")
    plt.show()


RANDOM = np.random.default_rng(0)


def examples(case_name, number=3, level=BY_PHOTO, model=MODELS[0]):
    history, _, catalogue, _ = DATA_BY_MODEL[model]
    by_line = {
        mode: {r["lines"][0]: r for r in level[mode, model] if case(r) == case_name}
        for mode in MODES
    }
    common = sorted(set.intersection(*(set(v) for v in by_line.values())))
    for line in RANDOM.choice(common, size=min(number, len(common)), replace=False):
        for mode in MODES:
            show(by_line[mode][line], history, catalogue, f"{mode} · ")


print("Bien rangé")
examples(CASES[0])
print("\nDans le top-10, mal rangé")
examples(CASES[1])
print("\nHors top-10")
examples(CASES[2])

### Comparaison `rank` (de chamfer.py) vs. `dorsalmatch.search`

`search` est un k-NN photo contre photo : il renvoie des paires (ligne du catalogue, cosinus). Chamfer compare des **ensembles** de photos et a besoin de la matrice complète des cosinus entre requête et candidat. Dans ce notebook, `rank` la calcule une fois et applique le mode choisi (`max` (k-NN de `dorsalmatch.search`), `chamfer_asym`, `chamfer_sym`), avec le même filtrage des côtés et des quasi-doublons que la baseline.

### Conclusions sur Chamfer

Le mode symétrique de Chamfer pénalise un indidivu qui a beaucoup de photos variées. Il faut regarder niveau "dauphin", où Chamfer a le plus de chances d'apporter quelque chose. En effet, au niveau photo, il n'y a aucune différence entre le `max` et la Chamfer distance (mêmes métriques). 

Néanmoins, niveau dauphin, le gain reste faible :
- `chamfer_sym` : 2 indidivus mieux, 0 moins bien, soit 52 bien rangés contre 51 pour max (88 % contre 86 %).
- `chamfer_asym` : 4 mieux, 2 moins bien, soit aussi 52 bien rangés contre 51. Le gain net est de 2 indidivus, mais il y a des régressions...

Le gain porte sur quelques indidivus déjà dans le top-10, qui sont mieux rangés. Il ne vient pas d'un indidivu retrouvé hors du top-10.

Cela veut dire que les faux positifs aberrants ne semblent pas être le facteur limitant, le `max()` étant sensible à ceux-ci. On pourrait garder le `chamfer_sym` par défaut... mais les indidivus sont trop peu nombreux (59) pour pouvoir avoir une réelle différence pour l'instant.

## Step 2 - Matching local / Re-ranking

Les erreurs restants sont principalement dues au mauvais classements dans le top-10. Le matching local compare 2 images point par point :
- ALIKED détecte des points clés et leurs descripteurs
- LightGlue établit des correspondances adaptatives entre les caractéristiques extraites, i.e. il apparie les points clés des deux images

Cela permet de faire du re-ranking : on a les embedding qui donne le top-K, puis le matching local qui reclasse ces top-K. Le matching local ne permet pas de récupérer un invididu hors de ce top-K.

Installer LightGlue : `uv pip install git+https://github.com/cvg/LightGlue.git`
Ajouter dans le groupe ml : `uv add --group ml git+https://github.com/cvg/LightGlue.git`

In [ ]:
from pathlib import Path

import numpy as np
import torch
from lightglue import ALIKED, LightGlue
from lightglue.utils import numpy_image_to_torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE", DEVICE)

ALIKED_MODEL = ALIKED(max_num_keypoints=2048).eval().to(DEVICE)
LIGHTGLUE_MODEL = LightGlue(features="aliked").eval().to(DEVICE)

Les modèles sont bien plus rapides en GPU qu'en CPU (+30mn vs. 3mn).

Pour passer en GPU/CUDA, il faut un GPU NVIDIA accessible avec une version CUDA de PyTorch compatible (`nvidia-smi` dans le cmd).

Il faut ensuite forcer la réinstallation depuis l'index CUDA : `uv pip install --reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu1XX` avec `cu1XX` qui correspond à la version CUDA appropriée, puis relancer le kernel.

In [ ]:
K = 10
MODE = "max"
ALPHA = 0.05
FEATURES = {}


def features(path):
    if path not in FEATURES:
        image = Image.open(path).convert("RGB")
        image.thumbnail((1024, 1024))
        tensor = numpy_image_to_torch(np.asarray(image)).to(DEVICE)
        with torch.no_grad():
            FEATURES[path] = ALIKED_MODEL.extract(tensor)
    return FEATURES[path]


def count_matches(path_a, path_b):
    with torch.no_grad():
        out = LIGHTGLUE_MODEL({"image0": features(path_a), "image1": features(path_b)})
    return len(out["matches"][0])


def local_score(query, individual):
    query_path = photo_path("historique", query["livraison"], query)
    return max(
        (
            count_matches(query_path, photo_path("catalogue", CATALOGUE, p))
            for p in catalogue
            if p["individu"] == individual and p["cote"] == query["cote"]
        ),
        default=0,
    )


def rerank(result):
    query = history[result["lines"][0]]
    top = result["candidats"][:K]
    scores = [local_score(query, c["individu"]) for c in top]
    order = np.argsort(scores)[::-1]
    ranked = [{**top[i], "matches": scores[i]} for i in order]
    labels = [c["individu"] for c in ranked]
    rank = (
        labels.index(result["individu"]) + 1 if result["individu"] in labels else None
    )
    return {**result, "candidats": ranked, "rang": rank, "rang_avant": result["rang"]}

In [ ]:
RERANKED = [rerank(r) for r in BY_PHOTO[MODE, reference]]
counts(RERANKED, "re-ranking local - par photo")

In [ ]:
pairs = [
    (r["rang"], r["rang_avant"]) for r in RERANKED if r["rang"] and r["rang_avant"]
]
new, old = np.array(pairs).T
print(
    f"mieux {np.sum(new < old)} / égal {np.sum(new == old)} / moins bien {np.sum(new > old)}"  # noqa: E501
)

### 2.1. Exemple de reranking positif

Le score local semble ici retrouver visuellement les points communs avec le bon individu.

In [ ]:
def examples_rerank(number):
    improved = [
        r
        for r in RERANKED
        if r["rang"] and r["rang_avant"] and r["rang"] < r["rang_avant"]
    ]
    for i in RANDOM.choice(
        len(improved), size=min(number, len(improved)), replace=False
    ):
        r = improved[i]
        show(r, history, catalogue, f"avant rang {r['rang_avant']} ")


examples_rerank(3)

### 2.2. Exemple de reranking négatif

Le score local semble ici se tromper (reflets, flou, mauvaise qualité...)

In [ ]:
def examples_degraded(number):
    degraded = [
        r
        for r in RERANKED
        if r["rang"] and r["rang_avant"] and r["rang"] > r["rang_avant"]
    ]
    for i in RANDOM.choice(
        len(degraded), size=min(number, len(degraded)), replace=False
    ):
        r = degraded[i]
        show(r, history, catalogue, f"avant rang {r['rang_avant']} ")


examples_degraded(3)

Pour l'instant, on a des dégradations mais meilleur top-10 : on a fait un re-ranking sans fusion. Cela vient de requêtes où le cosinus avait déjà le bon rang en général, la fusion devrait corriger...

Prochaines pistes :
- Utilisation de [RANSAC](https://www.geeksforgeeks.org/python/step-by-step-guide-to-using-ransac-in-opencv-using-python/) (homographie ou matrice fondamentale) : Cela permet la vérification géométrique en ne gardant que les appariements cohérents avec une même transformation; le nombre d'inliers sert de score (élevé pour 2 photos du même individu, faible pour 2 individus différents)
- Tester LoFTR (avec segmentation ?) 
- Tester la calibration/fusion du score pour chaque cas : combinaison du cosinus (global matchers) et du score local
- Tester les modèles MiewID & MegaDescriptor